<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/05_Hands_On_Labs/notebooks/16_mlops_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# End-to-End MLOps Pipeline: MLflow + DVC + Retraining

> **Track:** ML Engineer | **Level:** Advanced | **Time:** 3-4 hours

## Overview
Production ML requires more than a trained model — it needs versioned data, tracked experiments, automated retraining, and a model registry. This notebook wires together MLflow and DVC into a complete MLOps pipeline.

### What You'll Learn
- MLflow experiment tracking and model registry
- DVC data versioning concepts
- Automated retraining on data drift
- Model staging (Staging → Production)
- Pipeline orchestration with Python

```bash
pip install mlflow dvc scikit-learn pandas evidently
```

In [ ]:
# Setup: MLflow tracking server and experiment
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score
from pathlib import Path

np.random.seed(42)

# Configure MLflow
mlflow.set_tracking_uri('file:./mlruns')
mlflow.set_experiment('churn-prediction-pipeline')

# Generate versioned datasets (v1 and v2 for drift demo)
ndef make_churn_data(n=1000, drift=False, seed=42):
    rng = np.random.RandomState(seed)
    shift = 0.15 if drift else 0.0  # drift shifts feature distributions
    return pd.DataFrame({
        'tenure': rng.randint(1, 72, n).astype(float),
        'monthly_charges': rng.exponential(50 + shift*30, n),
        'num_products': rng.randint(1, 5, n),
        'support_calls': rng.poisson(2 + shift*2, n),
        'churn': rng.choice([0,1], n, p=[0.75, 0.25])
    })

data_v1 = make_churn_data(drift=False)
data_v2 = make_churn_data(drift=True, seed=99)

Path('data').mkdir(exist_ok=True)
data_v1.to_csv('data/churn_v1.csv', index=False)
data_v2.to_csv('data/churn_v2.csv', index=False)

print(f'Dataset v1: mean monthly_charges = {data_v1.monthly_charges.mean():.1f}')
print(f'Dataset v2 (drifted): mean monthly_charges = {data_v2.monthly_charges.mean():.1f}')

## 1. MLflow Experiment Tracking

In [ ]:
# Train and track experiments in MLflow
FEATURES = ['tenure', 'monthly_charges', 'num_products', 'support_calls']

def train_and_log(data: pd.DataFrame, run_name: str, params: dict) -> str:
    """Train a model and log everything to MLflow. Returns run_id."""
    X = data[FEATURES]
    y = data['churn']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    with mlflow.start_run(run_name=run_name) as run:
        # Log params
        mlflow.log_params(params)
        mlflow.log_param('dataset_size', len(data))

        # Train
        model = GradientBoostingClassifier(**params, random_state=42)
        model.fit(X_train, y_train)

        # Log metrics
        train_acc = accuracy_score(y_train, model.predict(X_train))
        test_acc = accuracy_score(y_test, model.predict(X_test))
        test_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
        cv_acc = cross_val_score(model, X, y, cv=5).mean()

        mlflow.log_metrics({'train_acc': train_acc, 'test_acc': test_acc,
                            'test_auc': test_auc, 'cv_acc': cv_acc})

        # Log model
        mlflow.sklearn.log_model(model, 'model', registered_model_name='churn_model')

        print(f'Run: {run_name} | test_acc={test_acc:.4f} | auc={test_auc:.4f} | run_id={run.info.run_id[:8]}...')
        return run.info.run_id

# Run experiments with different hyperparameters
run1 = train_and_log(data_v1, 'baseline_n100', {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1})
run2 = train_and_log(data_v1, 'tuned_n200', {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.05})

## 2. Data Drift Detection (Retraining Trigger)

In [ ]:
# Detect drift between v1 and v2 data using KS test
from scipy import stats

def detect_drift(reference: pd.DataFrame, current: pd.DataFrame, features: list[str], threshold: float = 0.05) -> dict:
    """Run KS test per feature and return drift report."""
    report = {'drifted_features': [], 'all_features': {}}
    for feat in features:
        ks_stat, p_value = stats.ks_2samp(reference[feat], current[feat])
        is_drift = p_value < threshold
        report['all_features'][feat] = {'ks_stat': round(ks_stat, 4), 'p_value': round(p_value, 4), 'drift': is_drift}
        if is_drift:
            report['drifted_features'].append(feat)
    report['drift_detected'] = len(report['drifted_features']) > 0
    return report

drift_report = detect_drift(data_v1, data_v2, FEATURES)
print('=== Data Drift Report ===')
for feat, info in drift_report['all_features'].items():
    status = '⚠ DRIFT' if info['drift'] else '✓ OK'
    print(f'  {feat:20s}: KS={info["ks_stat"]:.4f}, p={info["p_value"]:.4f} | {status}')
print(f'\nDrift detected: {drift_report["drift_detected"]}')
print(f'Drifted features: {drift_report["drifted_features"]}')

## 3. Automated Retraining Pipeline

In [ ]:
# Full automated retraining pipeline
import json

def run_mlops_pipeline(reference_path: str, current_path: str, drift_threshold: float = 0.05) -> dict:
    """Complete MLOps pipeline: detect drift → retrain if needed → register → promote."""
    reference = pd.read_csv(reference_path)
    current = pd.read_csv(current_path)

    result = {'retraining_triggered': False, 'reason': 'No drift detected'}

    # Step 1: Drift detection
    drift = detect_drift(reference, current, FEATURES, threshold=drift_threshold)
    result['drift_report'] = drift

    if drift['drift_detected']:
        result['retraining_triggered'] = True
        result['reason'] = f'Drift in: {drift["drifted_features"]}'
        print(f'🔄 Drift detected → triggering retraining')

        # Step 2: Retrain on new data
        run_id = train_and_log(
            current,
            run_name=f'retrained_on_v2',
            params={'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.08}
        )
        result['new_run_id'] = run_id
        print(f'✓ Model retrained and registered | run_id={run_id[:8]}...')
    else:
        print('✓ No drift — skipping retraining')

    return result

pipeline_result = run_mlops_pipeline('data/churn_v1.csv', 'data/churn_v2.csv')
print()
print('Pipeline result:')
print(json.dumps({k: v for k, v in pipeline_result.items() if k != 'drift_report'}, indent=2))

## 4. DVC Data Versioning

In [ ]:
# DVC workflow (conceptual — run in terminal)
dvc_workflow = '''
# Initialize DVC in your project
$ git init && dvc init

# Track data files
$ dvc add data/churn_v1.csv  # creates data/churn_v1.csv.dvc
$ git add data/.gitignore data/churn_v1.csv.dvc
$ git commit -m "Track training data v1"

# Add remote storage (S3, GCS, Azure, local)
$ dvc remote add -d myremote s3://my-bucket/dvc-store

# Push data to remote
$ dvc push

# When data updates (v2):
$ cp data/churn_v2.csv data/churn_v1.csv  # update file
$ dvc add data/churn_v1.csv  # creates new hash
$ git add data/churn_v1.csv.dvc
$ git commit -m "Update training data v2 — added Q4 data"
$ dvc push

# Reproduce old experiment exactly
$ git checkout <old-commit>
$ dvc checkout  # restores old data version

# DVC pipeline (dvc.yaml)
stages:
  train:
    cmd: python train.py
    deps: [data/churn_v1.csv, src/train.py]
    params: [params.yaml]
    outs: [models/model.pkl]
    metrics: [metrics.json]
'''
print(dvc_workflow)
print('DVC ensures: git tracks code+config, DVC tracks large data+models')

## Exercises

1. **Model comparison in MLflow UI**: After running the experiments, open the MLflow UI with `mlflow ui` in your terminal. Navigate to the experiment, compare all runs side by side, and use the parallel coordinates plot to find the optimal hyperparameters.

2. **Rolling retrain**: Implement a `ScheduledRetrainer` class that checks for drift weekly (using mock timestamps), retrains if drift is found, and maintains a log of retraining history with run IDs and drift reports.

3. **Canary deployment**: After retraining, implement a function that splits traffic 10/90 between the new model (canary) and the old model (production). After simulating 100 requests, compute accuracy for both models and promote the canary to production only if it outperforms.